# Fetch and Process Datasets

**Purpose:**
- Demonstrate where raw data comes from
- Call existing fetch and process modules
- Verify schemas used by demo notebooks

In [ ]:
import sys
from pathlib import Path

# Small shared root helper to handle path detection
_cwd = Path.cwd()
if _cwd.name != "notebooks":
    sys.path.insert(0, str(_cwd / "notebooks"))

import notebook_setup
ROOT = notebook_setup.setup_root()

import subprocess


## Configuration

In [ ]:
ALLOW_NETWORK_FETCH = False
REFRESH_EXISTING = False
DATASETS = ["gsm8k", "qmsum"]
SEED = 42


## Dataset Locations

In [ ]:
RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"
EVAL_DIR = ROOT / "data/evaluation"

print("Expected directories:")
print(f"RAW: {RAW_DIR}")
print(f"PROCESSED: {PROCESSED_DIR}")
print(f"EVAL: {EVAL_DIR}")


## Fetch and Process

In [ ]:
for ds in DATASETS:
    raw_path = RAW_DIR / ds
    eval_path = EVAL_DIR / f"{ds}_test.jsonl"
    
    if not eval_path.exists() or REFRESH_EXISTING:
        if not raw_path.exists():
            if ALLOW_NETWORK_FETCH:
                print(f"Fetching {ds}...")
                subprocess.run([sys.executable, f"scripts/eval_datasets/fetch_{ds}.py"], check=True)
            else:
                print(f"Instruction: {ds} raw data not found. Enable ALLOW_NETWORK_FETCH=True to download.")
                continue
                
        print(f"Processing {ds}...")
        subprocess.run([sys.executable, f"scripts/eval_datasets/process_{ds}.py"], check=True)
    else:
        print(f"Found existing evaluation set for {ds} at {eval_path.relative_to(ROOT)}. Skipping generation.")


## Audit and Preview

In [ ]:
import json

for ds in DATASETS:
    eval_path = EVAL_DIR / f"{ds}_test.jsonl"
    if not eval_path.exists():
        continue
        
    with open(eval_path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
        
    print(f"\n--- {ds.upper()} ---")
    print(f"Total evaluation rows: {len(rows)}")
    if rows:
        sample = rows[0]
        print(f"Schema keys: {list(sample.keys())}")
        print("Sample preview:")
        print(json.dumps(sample, indent=2)[:500] + "...")
